# mf_analyze (analyzing and presenting the GGOR outcomes from Modflow)

This file `mf_analyze.ipynob` is used to analyze and present the results of the mf6 simulation. It is
run separately after the simulation is complete.

Note that mf6 does not support `frf`, `fff` and `flf`, not even for the structured grid.
Therefore a separate python routine in `tools/fdm/src/mf6_face_flows.py` is used to compute those old-fashioned arrays from the cell-by-cell flows (cbc) file/the budget file, because contrary to the new JA data they are very transparent and easy to understand for the old-fashoned structured grid that we use in the GGOR tool.

@ TO 2025-07
"""

In [ ]:

import os

# ensure mf6lab/src, mf6lab/Projects/GGOR/src and .../<case>/src are in sys.path
import mf6_bootstrap # noqa: F401

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import flopy
from pathlib import Path
from mf6tools import Dirs

import time
from timing import log_timed

import etc
import ggor_tools as ggt
from settings import props

# --- setting up the logger
import logging
import logging_setup  # noqa: F401
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Start the actual script


In [ ]:

start_script = time.perf_counter()

case_name = Path(__file__).parent.parent.parts[-1]

dirs = Dirs()
dirs.meteo = os.path.join(Path(dirs.proj).parent, 'data', 'meteo')
dirs.bofek = os.path.join(Path(dirs.proj).parent, 'data', 'bofek')

# Read `tdata` and `parcel_data` back in and regenerate the `Grid` object

The `tdata` and `parcel_data` have been pickled in `mf_adapt` to prevent having to regenerate them in `mf_analyze` here. Instead they are just read in from their `.pkl` files.

In [ ]:

with log_timed(logger, 'Load gr, tdata, parcel_data'):
    # --- get saved tdata
    tdata_file = os.path.join(dirs.data, 'tdata.pkl')
    tdata = pd.read_pickle(tdata_file)

    # --- get saved parcel_data
    pdata_file = os.path.join(dirs.data, 'parcel_data.pkl')
    parcel_data = pd.read_pickle(pdata_file)

    # --- regenerate the grid object
    gr = ggt.grid_from_parcel_data(parcel_data=parcel_data,
                                   dx=props['dx'])

# Fetch the Modlfow simulation

In [ ]:

# --- Read the modflow simulation
with log_timed(logger, "loading MFsimulation"):
    sim = flopy.mf6.MFSimulation.load(sim_name=case_name,
                                  version='mf6',
                                  sim_ws=dirs.SIM,
                                  lazy_io=True)

# --- read the groundwater flow model
# with log_timed(logger, "sim.get_model"):
#    gwf = sim.get_model('{}Gwf'.format(sim.name).lower()) # list(sim.model_names)[0])


# Read Modflow's head file

The creates a heads object that can be questioned for its data.

The heads object also automatically computes the `HG3`, `VG3` and `LG3` values for every hydrological year (April 1 - March 31)in the `tdata` DataFrame, from which the GXG (`GHG`, `GVG`, `GLG`) are computed and added as columns to `parcel_data` GeoDataFrame.

In [ ]:

# --- Load the heads from Modflow and plot a parcel
with log_timed(logger, "heads_obj loaded"):
    heads_obj = ggt.Heads_obj(sim=sim, tdata=tdata, gr=gr)
    
    # --- plot time line for parcels
    # --- choose 1 or 2 ok, otherwise timeline plot becomes messy
    parcels = [0]
    title=['Parcel heads']
    axs = heads_obj.plot(tdata=tdata,                      
            parcel_data=parcel_data,
            parcels=parcels,
            plotGXG=True,
            figsize=(14, 8))
    axs[0].figure.suptitle(case_name)

# Read Moflow's budget file (cel by cell flows file)

This takes time because it is large. It yields a watbal object, the data of which can directly be plotted.

In [ ]:

# --- loading Modflows CBC output is expensive (takes about 50 seconds)
with log_timed(logger, "Watbal_obj loaded"):
    
    # --- Loading water budget components for all cells from CBC file
    print("Loading Watbal ... may take several minutes ...")
    
    watbal = ggt.Watbal_obj(sim=sim, dirs=dirs, gr=gr)

    # --- plot running water budget for all parcels
    axs = watbal.plot(parcel_data=parcel_data,
                        tdata=tdata,
                        parcels=None,   # over all parcels
                        sharey=True)
    axs[0].figure.suptitle(case_name)
    plt.gcf().savefig(dirs.images + '/watbal_all_parcels.png', dpi=300)

    # --- plot running water budget for all selected parcels
    axs = watbal.plot(parcel_data=parcel_data,
                        tdata=tdata,
                        parcels=parcels,   # id's of selected parcels
                        sharey=True)
    axs[0].figure.suptitle(case_name)
    plt.gcf().savefig(dirs.images + '/watbal.png', dpi=300)

# Plotting maps with the GHG, GVG and GLG

Parcel data is a geopandas.GeoDataFrame with polygons as its geometry, which is the circumference of the individual parcels. This allows to plot all the parcels and color them according the any column in the GeoDataFrame (parcel_data).

We generate three plots, one for GHG, one for GVG and one for GLG, i.e. the average highest groundwater level, the spring groundwate level and the average lowest groundwater level.

In [ ]:
# --- Plot 3 maps of the parcels GXG
with log_timed(logger, "GXG added, pickled and plotted"):

    vmin, vmax = np.inf, -np.inf
    for gxg in ['GHG', 'GVG', 'GLG']:
        parcel_data[gxg] = heads_obj.GXG[gxg]
        vmin = np.fmin(vmin, parcel_data[gxg].min())
        vmax = np.fmax(vmax, parcel_data[gxg].max())
        
    # --- overwite old parcel_data.pkl file
    parcel_data.to_pickle(pdata_file)

    # --- map GHG, GVG, GLG
    for what in ['GHG', 'GVG', 'GLG']:
        ax = etc.newfig(f"{what} {case_name}", "xRD", "yRD",
                        figsize=(10, 8))
        # --- plot map of G?G of parcels
        parcel_data.plot(what,
                     cmap='viridis',
                     vmin=vmin,
                     vmax=vmax,
                     legend=True,
                     edgecolor='black',
                     figsize=(10, 8),
                     ax=ax)
    ax.figure.suptitle(case_name)
    plt.gcf().savefig(dirs.images + f"/map_{what}.png", dpi=300)
                     

plt.show()


In [ ]:

# --- total elapsed time in script
logger.info(f"mf_analyse finished in {time.perf_counter()-start_script:.2f} seconds")
